# 01 Cleaning
Objective: load the raw inspections file, clean it, and save a processed dataset plus a one-row-per-violation table.

In [1]:
import pandas as pd
import numpy as np

raw_path = "../data/raw/food_inspections.csv"
df = pd.read_csv(raw_path)
df.shape

(317225, 17)

## Load the data
I loaded the raw CSV and checked its shape. The dataset has 317,225 rows and 17 columns, and each row is one food inspection in Chicago.

# 02 Inspect the columns
Look at columns, types, missing values, and duplicates.

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 317225 entries, 0 to 317224
Data columns (total 17 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Inspection ID    317225 non-null  int64  
 1   DBA Name         317225 non-null  object 
 2   AKA Name         314801 non-null  object 
 3   License #        317206 non-null  float64
 4   Facility Type    311868 non-null  object 
 5   Risk             317138 non-null  object 
 6   Address          317225 non-null  object 
 7   City             317043 non-null  object 
 8   State            317149 non-null  object 
 9   Zip              317183 non-null  float64
 10  Inspection Date  317225 non-null  object 
 11  Inspection Type  317224 non-null  object 
 12  Results          317225 non-null  object 
 13  Violations       227719 non-null  object 
 14  Latitude         316163 non-null  float64
 15  Longitude        316163 non-null  float64
 16  Location         316163 non-null  obje

# Inspect the columns
Most of the 17 columns are text. Three things need fixing:
- Inspection Date is stored as text, so I will convert it to a date type.
- License # and Zip are stored as decimal numbers (float). They are identifiers, not quantities, so I will convert them to text.
- Column names contain spaces, capital letters and a "#" symbol, so I will rename them to lowercase with underscores.
The non-null counts match my missing-values check. Violations is missing in 89,506 rows (317,225 minus 227,719).

# Clean the column names

In [3]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("#","")

#cleaned white spaces before and after the string, converted to lower, replaced spaces with _ and replaced #.

In [4]:
df.head()

,inspection_id,dba_name,aka_name,license_,facility_type,risk,address,city,state,zip,inspection_date,inspection_type,results,violations,latitude,longitude,location
0,2645405,BURRITO BEACH,BURRITO BEACH,2966529.0,Restaurant,Risk 1 (High),5700 S MARYLAND AVE,CHICAGO,IL,60637.0,10/19/2026,Canvass,Pass,"38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - ...",41.791261,-87.605085,"(41.79126061783551, -87.60508491200589)"
1,2645427,"GRAHAM CRACKER'S DAYCARE CENTER II, INC","GRAHAM CRACKER'S DAYCARE CENTER II, INC.",2632026.0,Children's Services Facility,Risk 1 (High),1 E 113TH ST,CHICAGO,IL,60628.0,10/19/2026,Canvass,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,41.688848,-87.622894,"(41.688848119400404, -87.6228936894908)"
2,2645374,SIP & SAVOR COFFEE HOUSE,SIP & SAVOR,3002330.0,Restaurant,Risk 1 (High),528 E 43RD ST,CHICAGO,IL,60653.0,10/19/2026,Canvass Re-Inspection,Pass w/ Conditions,"1. PERSON IN CHARGE PRESENT, DEMONSTRATES KNOW...",41.816815,-87.612923,"(41.816815018289546, -87.61292308115752)"
3,2645430,BENTO,BENTO SUSHI,3020653.0,Restaurant,Risk 1 (High),5700 S MARYLAND AVE,CHICAGO,IL,60637.0,10/19/2026,Canvass,Pass,NaN,41.791261,-87.605085,"(41.79126061783551, -87.60508491200589)"
4,2645408,"BELLA EATERIES, L.L.C.",CHICK-FIL-A,3056989.0,Restaurant,Risk 1 (High),1512 E 53RD ST,CHICAGO,IL,60615.0,10/19/2026,Canvass,Pass,"55. PHYSICAL FACILITIES INSTALLED, MAINTAINED ...",41.799658,-87.588578,"(41.79965791698844, -87.58857814228286)"


In [5]:
df.isna().sum().sort_values(ascending=False)

violations         89506
facility_type       5357
aka_name            2424
location            1062
longitude           1062
latitude            1062
city                 182
risk                  87
state                 76
zip                   42
license_              19
inspection_type        1
inspection_date        0
results                0
dba_name               0
address                0
inspection_id          0
dtype: int64

## rename license_ to license_number

In [6]:
df = df.rename(columns={"license_":"license_number"})

In [7]:
df.head(2)

,inspection_id,dba_name,aka_name,license_number,facility_type,risk,address,city,state,zip,inspection_date,inspection_type,results,violations,latitude,longitude,location
0,2645405,BURRITO BEACH,BURRITO BEACH,2966529.0,Restaurant,Risk 1 (High),5700 S MARYLAND AVE,CHICAGO,IL,60637.0,10/19/2026,Canvass,Pass,"38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - ...",41.791261,-87.605085,"(41.79126061783551, -87.60508491200589)"
1,2645427,"GRAHAM CRACKER'S DAYCARE CENTER II, INC","GRAHAM CRACKER'S DAYCARE CENTER II, INC.",2632026.0,Children's Services Facility,Risk 1 (High),1 E 113TH ST,CHICAGO,IL,60628.0,10/19/2026,Canvass,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,41.688848,-87.622894,"(41.688848119400404, -87.6228936894908)"


## Check duplicates

In [8]:
df.duplicated().sum()

np.int64(0)

In [10]:
##There are no duplicated in the dataset

## Checking duplicate inspection ids

In [ ]:
df["inspection_id"].duplicated().sum()

No duplicate inspection ids found.

## Dates and text
Convert the date, tidy text columns, and look at the result values.

In [ ]:
df["dba_name"] = df["dba_name"].astype("string").str.strip()
df["inspection_date"] = pd.to_datetime(df["inspection_date"], errors="coerce")
for col in ["facility_type", "risk", "results"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip().str.title()
df["results"].value_counts()

## Fix data types and standardize text
Converted inspection_date to a datetime and standardized the capitalization of facility_type, risk and results so the same value is not counted twice. I kept dba_name's original capitalization because business names like "McDonald's" would be damaged. The results column now has 7 clean categories, led by Pass (164,230) and Fail (61,090).

In [ ]:
df[df["violations"].isna()]["results"].value_counts()

In [ ]:
#### Missing violations
I checked the results for rows with no violations recorded. About 49% were inspections that did not happen (Out Of Business, No Entry, Not Ready, 
Business Not Located) and about 46% were Pass, so an empty Violations field mostly means nothing was found. A small group of 4,606 rows (about 5%) are 
Fail or Pass W/ Conditions with no violations listed, which may be a gap in the source data. I kept all of these rows and will handle the violations 
text when I split it.

## Step 4: Create analysis columns
Decide what counts as a fail. Here only the result 'Fail' does; write down your choice and why.

In [ ]:
df["is_fail"] = df["results"].eq("Fail")
df["inspection_year"] = df["inspection_date"].dt.year
df["inspection_month"] = df["inspection_date"].dt.month
df[["results", "is_fail", "inspection_year"]].head()

## Step 5: Split the violations text
Each inspection has many violations in one cell, separated by `|`. Make one row per violation.

In [ ]:
viol = df[["inspection_id", "violations"]].dropna().copy()
viol["violations"] = viol["violations"].str.split("|")
viol = viol.explode("violations")
viol["violations"] = viol["violations"].str.strip()

# Pull out the violation number and description; drop the inspector's comments
extracted = viol["violations"].str.extract(r"^(\d+)\.\s*(.*?)(?:\s*-\s*Comments:.*)?$", expand=True)
viol["violation_code"] = pd.to_numeric(extracted[0], errors="coerce")
viol["violation_desc"] = extracted[1].str.title()
viol = viol.drop(columns="violations").dropna(subset=["violation_code"])
viol.head()

## Step 6: Check and save
Check row counts and a few random rows before saving.

In [ ]:
keep = ["inspection_id", "dba_name", "license", "facility_type", "risk", "zip",
        "inspection_date", "inspection_type", "results", "is_fail",
        "latitude", "longitude"]
keep = [c for c in keep if c in df.columns]
inspections = df[keep].drop_duplicates("inspection_id").rename(columns={"license": "license_no"})

inspections.to_csv("../data/processed/inspections_clean.csv", index=False)
viol.to_csv("../data/processed/violations_clean.csv", index=False)
inspections.shape, viol.shape

## What I learned / data issues I found
- 